# FORGE — pipeline di produzione: MySQL → forge() → validazione hold-out → persistenza regole

Questo notebook esegue l'intero ciclo per scoprire regole di trading con **forgedge/FORGE** in modo
disciplinato dal punto di vista statistico:

1. **Ingestione** delle candele da MySQL (parametrizzata per `ticker` e `timeframe`).
2. **Censura** della serie storica: l'ultima porzione (periodo di hold-out) viene esclusa da ogni fase
   di discovery/selezione dei parametri — non la vede nessuno fino al punto 4.
3. **Esecuzione di `forge()`** sul solo periodo di training, eventualmente con più preset
   (`sniper` / `balanced` / `sweep` / `burst`), così da coprire profili di ricerca diversi.
4. **Validazione out-of-sample** di ogni regola sopravvissuta (verdetto `EDGE`/`PARTIAL-EDGE`) nel
   periodo di hold-out, usando il **motore di backtest integrato di forge** (`forgedge.rule_discovery.run_backtest`)
   sugli stessi identici parametri operativi scelti in fase di training — nessuna riselezione dei
   parametri sul hold-out, per evitare qualunque leakage.
5. **Persistenza** di ogni regola che supera anche il hold-out: un file **JSON** con i parametri
   operativi (direzione, `sell_pct`, `target_h`, ...) e le metriche di performance (win rate, profit
   factor, expectancy, ...) sia in-sample/walk-forward che di hold-out.

Il notebook è testato sulla fixture `tests/fixtures/ADA_1D_TRAIN.parquet` (candele giornaliere ADA) —
impostare `DATA_SOURCE = "mysql"` per usarlo in produzione con un vero database.

> **Invariante da rispettare**: il periodo di hold-out non deve mai influenzare le soglie degli eventi
> (M1), il target derivato (M2) o la selezione dei parametri operativi (M3). Viene usato *soltanto* per
> un singolo backtest finale, a parametri congelati — esattamente il pattern "monitorare un edge
> pubblicato su dati nuovi" descritto nella skill `forgedge` (pattern 5), applicato qui al proprio
> hold-out invece che a candele realmente future.


## 0. Setup

In [1]:
import sys
sys.path.insert(0, "../src")   # per esecuzione da notebooks/

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from forgedge import forge, forge_preset, config_report, summary_report
from forgedge.rule_discovery import run_backtest

pd.set_option("display.width", 140)


## 1. Configurazione

Tutti i parametri della run sono qui. `DATA_SOURCE` seleziona la sorgente dati:

- `"mysql"` — candele grezze lette da MySQL, parametrizzate per `TICKER`/`TIMEFRAME`; viene poi
  costruita la KPI Table con `forgedge.build_features`/`candle_features`.
- `"fixture"` — usa direttamente la fixture di test del repo (`tests/fixtures/ADA_1D_TRAIN.parquet`),
  già una KPI Table pronta. Usata per testare il notebook end-to-end senza un database.


In [2]:
# --- sorgente dati ----------------------------------------------------------
DATA_SOURCE = "fixture"          # "mysql" | "fixture"
TICKER      = "ADAUSDC"
TIMEFRAME   = "1D"               # "1H" o "1D" — i due timeframe supportati da questo notebook

# --- connessione MySQL (usata solo se DATA_SOURCE == "mysql") ---------------
MYSQL_CONFIG = dict(
    host="localhost",
    port=3306,
    user="forge",
    password="",          # preferire una variabile d'ambiente in produzione
    database="market_data",
)
# Nome tabella per timeframe — adattare allo schema del proprio DB.
MYSQL_TABLE_BY_TIMEFRAME = {
    "1H": "candles_1h",
    "1D": "candles_1d",
}

# --- fixture di test (usata solo se DATA_SOURCE == "fixture") ---------------
FIXTURE_PATH = "../tests/fixtures/ADA_1D_TRAIN.parquet"

# --- hold-out -----------------------------------------------------------
HOLDOUT_FRACTION = 0.20   # ultima frazione della storia censurata dalla discovery

# --- preset da provare in sequenza (vedi skill forgedge: 'sniper' e 'sweep'
# richiedono storie lunghe su 1D e possono essere scartati dal coherence check) ---
PRESETS = ["burst", "balanced", "sniper", "sweep"]

# --- soglie di sopravvivenza sul hold-out ------------------------------------
# NOTE: BacktestSummary.win_rate_pct è una FRAZIONE 0-1 nonostante il nome.
MIN_HOLDOUT_TRADES          = 5
MIN_HOLDOUT_PROFIT_FACTOR   = 1.0
MIN_HOLDOUT_WIN_RATE        = 0.45

# --- output -------------------------------------------------------------
OUTPUT_DIR = Path("./discovered_rules")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Ingestione candele da MySQL

Query parametrizzata su `ticker`/`timeframe`. Richiede `sqlalchemy` + `pymysql`
(`pip install sqlalchemy pymysql`). Lo schema atteso per la tabella candele è
`(open_time, open, high, low, close, volume, symbol)` — adattare la query se il proprio schema
differisce. `open_time` può essere epoch (ms/s) o già un datetime: la conversione è gestita sotto.


In [3]:
def load_candles_from_mysql(ticker: str, timeframe: str, config: dict = MYSQL_CONFIG) -> pd.DataFrame:
    """Carica candele OHLCV grezze da MySQL per un ticker/timeframe, ordinate cronologicamente."""
    from sqlalchemy import create_engine, text

    table = MYSQL_TABLE_BY_TIMEFRAME.get(timeframe)
    if table is None:
        raise ValueError(f"Nessuna tabella configurata per timeframe={timeframe!r} "
                          f"(vedi MYSQL_TABLE_BY_TIMEFRAME)")

    conn_str = (
        f"mysql+pymysql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['database']}"
    )
    engine = create_engine(conn_str)
    query = text(f"""
        SELECT open_time, open, high, low, close, volume
        FROM {table}
        WHERE symbol = :ticker
        ORDER BY open_time ASC
    """)
    with engine.connect() as conn:
        raw = pd.read_sql(query, conn, params={"ticker": ticker})

    if raw.empty:
        raise RuntimeError(f"Nessuna candela trovata per {ticker} / {timeframe} in {table}")

    # open_time può essere epoch ms, epoch s o già datetime: normalizza.
    if np.issubdtype(raw["open_time"].dtype, np.number):
        unit = "ms" if raw["open_time"].iloc[0] > 10**12 else "s"
        raw["open_dt"] = pd.to_datetime(raw["open_time"], unit=unit)
    else:
        raw["open_dt"] = pd.to_datetime(raw["open_time"])

    raw = raw.drop(columns=["open_time"]).sort_values("open_dt").reset_index(drop=True)
    return raw


def load_fixture(path: str) -> pd.DataFrame:
    """Carica la fixture di test (già una KPI Table completa, non candele grezze)."""
    df = pd.read_parquet(path)
    return df.sort_values("open_dt").reset_index(drop=True)


if DATA_SOURCE == "mysql":
    raw_candles = load_candles_from_mysql(TICKER, TIMEFRAME)
    print(f"MySQL: {len(raw_candles)} candele {TICKER}/{TIMEFRAME} "
          f"({raw_candles['open_dt'].min()} -> {raw_candles['open_dt'].max()})")
elif DATA_SOURCE == "fixture":
    raw_candles = load_fixture(FIXTURE_PATH)
    print(f"Fixture: {len(raw_candles)} righe ({raw_candles['open_dt'].min()} -> {raw_candles['open_dt'].max()})")
else:
    raise ValueError(f"DATA_SOURCE sconosciuta: {DATA_SOURCE!r}")


Fixture: 882 righe (2024-01-01 00:00:00 -> 2026-05-31 00:00:00)


## 3. Costruzione della KPI Table

Se la sorgente è MySQL (candele grezze OHLCV) costruiamo gli indicatori con `build_features`/
`candle_features` di forgedge. Se la sorgente è la fixture di test, è già una KPI Table pronta e
questo passo viene saltato. `summary_report()` valida la qualità dei dati prima di procedere
(non blocca mai da sola — decidiamo noi cosa fare con `has_critical`).


In [4]:
if DATA_SOURCE == "mysql":
    from forgedge import build_features, candle_features

    kpi = build_features(raw_candles, timestamp_col="open_dt")
    kpi = candle_features(kpi)
else:
    kpi = raw_candles

rep = summary_report(kpi, timeframe=TIMEFRAME, return_report=True, verbose=False)
print(rep.one_line() if hasattr(rep, "one_line") else rep)
if rep.has_critical:
    raise ValueError("La KPI Table ha problemi critici di qualità — correggere prima di procedere.")

print(f"\nKPI table: {len(kpi)} bar, {kpi['open_dt'].min()} -> {kpi['open_dt'].max()}, "
      f"{kpi.shape[1]} colonne")


[WARN] OHLC: range intrabar >100% (max/min>2) in 1 barre; [WARN] 1 rendimenti anomali (MAD z>8) — possibili print errati o salti di scala; [WARN] 1 barre con |rendimento| > 50% — estremo; reale su asset molto volatili (es. crypto), ma da verificare se non atteso

KPI table: 882 bar, 2024-01-01 00:00:00 -> 2026-05-31 00:00:00, 26 colonne


## 4. Censura del periodo di hold-out

Si taglia la serie cronologicamente: l'ultima `HOLDOUT_FRACTION` dei bar non viene mai passata a
`forge()` — né come dato di training né come dato di selezione (M1/M2/M3 operano solo su `train_df`).
Verrà riunita a `train_df` **soltanto** al passo 6, per un singolo backtest finale a parametri
congelati (mai per ri-allenare soglie o target).


In [5]:
n = len(kpi)
split_idx = int(round(n * (1 - HOLDOUT_FRACTION)))

train_df = kpi.iloc[:split_idx].reset_index(drop=True)
holdout_df = kpi.iloc[split_idx:].reset_index(drop=True)
holdout_start = holdout_df["open_dt"].iloc[0]

print(f"train   : {len(train_df):4d} bar  ({train_df['open_dt'].iloc[0]} -> {train_df['open_dt'].iloc[-1]})")
print(f"hold-out: {len(holdout_df):4d} bar  ({holdout_start} -> {holdout_df['open_dt'].iloc[-1]})")


train   :  706 bar  (2024-01-01 00:00:00 -> 2025-12-06 00:00:00)
hold-out:  176 bar  (2025-12-07 00:00:00 -> 2026-05-31 00:00:00)


## 5. Esecuzione di `forge()` sul solo training, per ogni preset

Per ciascun preset: si costruisce la tripla di config con `forge_preset()`, si verifica la coerenza
con `config_report()` **prima** di lanciare la pipeline (pattern 6 della skill `forgedge`) e si salta
il preset se il check trova un `FAIL` — su una storia 1D corta (come questa fixture dopo aver tolto
l'hold-out) è normale che `sniper`/`sweep` vengano scartati (vedi pitfall #8 della skill): non è un
bug, è il controllo di coerenza che rifiuta correttamente una sessione che non potrebbe sostenere il
test richiesto.


In [6]:
preset_results = {}

for preset in PRESETS:
    disc_cfg, alpha_cfg, rd_cfg = forge_preset(preset, timeframe=TIMEFRAME, asset=TICKER)
    rep = config_report(disc_cfg, alpha_cfg, rd_cfg, kpi=train_df, timeframe=TIMEFRAME)

    if rep.has_critical:
        print(f"[{preset:10s}] SKIPPED — coherence FAIL: {rep.one_line()}")
        continue

    print(f"[{preset:10s}] coherence OK — running forge()...")
    try:
        result = forge(
            train_df,
            ticker=TICKER,
            timeframe=TIMEFRAME,
            event_discovery_config=disc_cfg,
            alpha_config=alpha_cfg,
            rule_discovery_config=rd_cfg,
            progress=False,
        )
    except ValueError as exc:
        print(f"[{preset:10s}] forge() raised: {exc}")
        continue

    n_edges = sum(1 for _, r in result.rule_responses if r.is_edge)
    print(f"[{preset:10s}] done — {len(result.rule_responses)} regole valutate, {n_edges} tradeable")
    preset_results[preset] = result

if not preset_results:
    raise RuntimeError("Nessun preset ha prodotto una run coerente su questa finestra di training.")


[burst     ] coherence OK — running forge()...


[burst     ] done — 305 regole valutate, 5 tradeable
[balanced  ] coherence OK — running forge()...


[balanced  ] done — 853 regole valutate, 94 tradeable
[sniper    ] SKIPPED — coherence FAIL: [WARN] m1_is_window_too_short: la finestra di discovery IS (23.5 mesi = 23.5 mesi di storia x train_ratio=1) non raggiunge i gate_params.min_episodes=10 al tasso configurato min_tpm=0.3: servono 53.3 mesi con margine di Poisson al 95 % (4.44 anni). Un candidato esattamente al floor di frequenza non avrà mai abbastanza episodi per superare il Criterion 2 su questa storia — non un errore, ma la ragione per cui una IS lunga rende meno di quanto sembri: abbassare min_episodes a <= 4, oppure alzare min_tpm, oppure allungare la storia.; [FAIL] oos_span_too_short: walk_forward.min_train_months=31 copre l'intero span disponibile (24 mesi): non resta nessuna finestra di test. La finestra è derivata da criteria.min_tpm=0.528 — servono 10 trade con margine di Poisson 95 % (#173). Alzare min_tpm a >= 1.36 perché selezione e test stiano entrambi nello span, oppure allungare la storia.
[sweep     ] SKIPPED —

## 6. Raccolta e deduplica delle regole tradeable tra preset

Si raccolgono tutti i responsi con verdetto `EDGE`/`PARTIAL-EDGE` da ogni preset eseguito. Più
preset possono riscoprire pattern simili: la deduplica è fatta su `(direzione, event_expression)`,
tenendo — a parità di pattern — quello con miglior profit factor walk-forward OOS **di training**
(il hold-out non entra qui).


In [7]:
# mappa event_id -> EventCandidate, serve più avanti per ricostruire il segnale sul hold-out
candidates_by_id = {}
for result in preset_results.values():
    for cand in result.candidates:
        candidates_by_id[cand.event_id] = cand

pool = {}  # (direzione, expression) -> (preset, contract, response, oos_pf)
for preset, result in preset_results.items():
    for contract, response in result.rule_responses:
        if not response.is_edge:
            continue
        key = (contract.derived_target.direction, contract.event_expression)
        oos_pf = (
            response.walk_forward.oos_summary.profit_factor
            if response.walk_forward else float("nan")
        )
        prev = pool.get(key)
        if prev is None or (not np.isnan(oos_pf) and oos_pf > prev[3]):
            pool[key] = (preset, contract, response, oos_pf)

print(f"{len(pool)} regole tradeable distinte su {len(preset_results)} preset (pre hold-out)")


98 regole tradeable distinte su 2 preset (pre hold-out)


## 7. Validazione hold-out con il backtest integrato di forge

Per ogni regola del pool: si ricostruisce il segnale booleano sull'intera serie (train + hold-out)
con `EventCandidate.apply()` — la stessa identica logica di soglie fissata in fase di discovery — e
si esegue **`forgedge.rule_discovery.run_backtest`** (il motore di backtest di Modulo 3) ristretto
alle sole barre di apertura del periodo di hold-out (`timerange_from=holdout_start`), usando
esattamente i parametri operativi già scelti in training (`response.validated_rule.params`).

Nessuna riottimizzazione sul hold-out: è un singolo backtest a parametri congelati, esattamente il
pattern di monitoraggio raccomandato dalla skill `forgedge` per un edge già pubblicato.


In [8]:
full_df = pd.concat([train_df, holdout_df], ignore_index=True).sort_values("open_dt").reset_index(drop=True)
SIGNAL_COL = "__holdout_signal__"

holdout_results = []  # lista di dict con tutto il necessario per il passo 8

for (direction, expr), (preset, contract, response, train_oos_pf) in pool.items():
    candidate = candidates_by_id.get(contract.event_candidate_id)
    if candidate is None:
        print(f"SKIP {contract.alpha_id}: EventCandidate non trovato nel pool")
        continue

    params = response.validated_rule.params

    work = full_df.copy()
    work[SIGNAL_COL] = candidate.apply(work).fillna(0).astype(bool)

    holdout_summary, holdout_trades = run_backtest(
        work, SIGNAL_COL, params,
        timerange_from=str(holdout_start), timerange_to=None,
        timestamp_col="open_dt", return_trades=True,
    )

    survives = (
        holdout_summary.total_trades >= MIN_HOLDOUT_TRADES
        and holdout_summary.profit_factor >= MIN_HOLDOUT_PROFIT_FACTOR
        and holdout_summary.win_rate_pct >= MIN_HOLDOUT_WIN_RATE
    )

    print(f"{contract.alpha_id:30s} [{preset:9s}] train={response.verdict:13s} -> "
          f"holdout: trades={holdout_summary.total_trades:3d} "
          f"pf={holdout_summary.profit_factor:6.2f} "
          f"win_rate={holdout_summary.win_rate_pct:5.1%} "
          f"exp={holdout_summary.expectancy:+.4f}  "
          f"-> {'SURVIVES' if survives else 'rejected'}")

    holdout_results.append(dict(
        preset=preset, contract=contract, response=response, candidate=candidate,
        holdout_summary=holdout_summary, holdout_trades=holdout_trades, survives=survives,
    ))

survivors = [r for r in holdout_results if r["survives"]]
print(f"\n{len(survivors)}/{len(holdout_results)} regole sopravvivono alla validazione hold-out")


ALPHA-ADAUSDC-1D-261001-329    [burst    ] train=PARTIAL-EDGE  -> holdout: trades= 17 pf=  0.52 win_rate=35.3% exp=-0.0250  -> rejected
ALPHA-ADAUSDC-1D-261001-766    [burst    ] train=PARTIAL-EDGE  -> holdout: trades= 24 pf=  0.37 win_rate=50.0% exp=-0.0337  -> rejected
ALPHA-ADAUSDC-1D-261001-885    [burst    ] train=PARTIAL-EDGE  -> holdout: trades=  9 pf=  0.30 win_rate=33.3% exp=-0.0450  -> rejected
ALPHA-ADAUSDC-1D-261001-1155   [burst    ] train=PARTIAL-EDGE  -> holdout: trades= 13 pf=  0.23 win_rate=30.8% exp=-0.0204  -> rejected
ALPHA-ADAUSDC-1D-261001-1165   [burst    ] train=PARTIAL-EDGE  -> holdout: trades= 13 pf=  0.23 win_rate=30.8% exp=-0.0204  -> rejected
ALPHA-ADAUSDC-1D-261001-070    [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  3 pf=  2.33 win_rate=33.3% exp=+0.0258  -> rejected
ALPHA-ADAUSDC-1D-261001-135    [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  1 pf=9999.00 win_rate=100.0% exp=+0.0793  -> rejected
ALPHA-ADAUSDC-1D-261001-149    [balanced ] tra

ALPHA-ADAUSDC-1D-261001-1978   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  9 pf=  0.05 win_rate=11.1% exp=-0.0286  -> rejected
ALPHA-ADAUSDC-1D-261001-2012   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  3 pf=  0.05 win_rate=33.3% exp=-0.0236  -> rejected
ALPHA-ADAUSDC-1D-261001-2014   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  2 pf=  0.00 win_rate= 0.0% exp=-0.0242  -> rejected
ALPHA-ADAUSDC-1D-261001-2037   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  3 pf=  0.05 win_rate=33.3% exp=-0.0236  -> rejected
ALPHA-ADAUSDC-1D-261001-2052   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  8 pf=  0.71 win_rate=62.5% exp=-0.0165  -> rejected


ALPHA-ADAUSDC-1D-261001-2082   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  2 pf=  0.00 win_rate= 0.0% exp=-0.0193  -> rejected
ALPHA-ADAUSDC-1D-261001-2083   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  2 pf=  0.00 win_rate= 0.0% exp=-0.0242  -> rejected
ALPHA-ADAUSDC-1D-261001-2086   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  2 pf=  0.00 win_rate= 0.0% exp=-0.0242  -> rejected
ALPHA-ADAUSDC-1D-261001-2098   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  4 pf=9999.00 win_rate=100.0% exp=+0.0352  -> rejected
ALPHA-ADAUSDC-1D-261001-2101   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  2 pf=  0.00 win_rate= 0.0% exp=-0.0193  -> rejected


ALPHA-ADAUSDC-1D-261001-2114   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  4 pf=9999.00 win_rate=100.0% exp=+0.0352  -> rejected
ALPHA-ADAUSDC-1D-261001-2121   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  3 pf=  0.05 win_rate=33.3% exp=-0.0236  -> rejected
ALPHA-ADAUSDC-1D-261001-2138   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  8 pf=  0.66 win_rate=62.5% exp=-0.0193  -> rejected
ALPHA-ADAUSDC-1D-261001-2182   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  1 pf=9999.00 win_rate=100.0% exp=+0.0903  -> rejected
ALPHA-ADAUSDC-1D-261001-2195   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  2 pf=  0.00 win_rate= 0.0% exp=-0.0535  -> rejected
ALPHA-ADAUSDC-1D-261001-2198   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  1 pf=  0.00 win_rate= 0.0% exp=-0.0420  -> rejected
ALPHA-ADAUSDC-1D-261001-2225   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  7 pf=  1.04 win_rate=42.9% exp=+0.0009  -> rejected
ALPHA-ADAUSDC-1D-261001-2380   [balanced ] t

ALPHA-ADAUSDC-1D-261001-2549   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  7 pf=  0.13 win_rate=14.3% exp=-0.0247  -> rejected
ALPHA-ADAUSDC-1D-261001-2594   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  0 pf=  0.00 win_rate= nan% exp=+nan  -> rejected
ALPHA-ADAUSDC-1D-261001-2609   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  1 pf=  0.00 win_rate= 0.0% exp=-0.0371  -> rejected
ALPHA-ADAUSDC-1D-261001-2610   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  1 pf=  0.00 win_rate= 0.0% exp=-0.0520  -> rejected


ALPHA-ADAUSDC-1D-261001-2645   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  3 pf=  1.15 win_rate=33.3% exp=+0.0038  -> rejected
ALPHA-ADAUSDC-1D-261001-2679   [balanced ] train=PARTIAL-EDGE  -> holdout: trades=  4 pf=  0.00 win_rate= 0.0% exp=-0.0439  -> rejected

10/98 regole sopravvivono alla validazione hold-out


## 8. Persistenza delle regole sopravvissute

Per ogni regola sopravvissuta scriviamo un JSON con:

- **parametri operativi**: espressione dell'evento, direzione, modalità di entrata, `buy_drop_pct`,
  `buy_delay_bar`, `sell_pct`, `target_h`, fee (da `ValidatedRule.to_dict()`);
- **parametri di performance**: metriche complete (`total_trades`, `profit_factor`, `win_rate_pct`,
  `expectancy`, `tpm_mu`, concorrenza, ...) sia per il training (in-sample e walk-forward OOS) sia per
  il backtest di hold-out appena eseguito.

Viene scritto anche un indice CSV riassuntivo con una riga per regola sopravvissuta.


In [9]:
def _clean(obj):
    """Rende l'oggetto serializzabile in JSON: NaN -> null, numpy -> python nativi."""
    if isinstance(obj, dict):
        return {k: _clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_clean(v) for v in obj]
    if isinstance(obj, float) and np.isnan(obj):
        return None
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    return obj


index_rows = []

for r in survivors:
    contract, response = r["contract"], r["response"]

    record = {
        "rule_id": contract.alpha_id,
        "ticker": TICKER,
        "timeframe": TIMEFRAME,
        "preset": r["preset"],
        "verdict_train": response.verdict,
        "event_expression": contract.event_expression,
        "operational_params": response.validated_rule.to_dict(),
        "performance": {
            "train_in_sample": response.in_sample_summary.to_dict(),
            "train_walk_forward_oos": (
                response.walk_forward.oos_summary.to_dict() if response.walk_forward else None
            ),
            "holdout": r["holdout_summary"].to_dict(),
        },
        "holdout_window": {
            "from": str(holdout_start),
            "to": str(holdout_df["open_dt"].iloc[-1]),
        },
    }
    record = _clean(record)

    out_path = OUTPUT_DIR / f"{contract.alpha_id}.json"
    out_path.write_text(json.dumps(record, indent=2, default=str))
    print("scritto", out_path)

    index_rows.append({
        "rule_id": contract.alpha_id,
        "preset": r["preset"],
        "direction": response.validated_rule.params.direction,
        "verdict_train": response.verdict,
        "target_h": response.validated_rule.params.target_h,
        "sell_pct": response.validated_rule.params.sell_pct,
        "holdout_trades": r["holdout_summary"].total_trades,
        "holdout_profit_factor": r["holdout_summary"].profit_factor,
        "holdout_win_rate": r["holdout_summary"].win_rate_pct,
        "holdout_expectancy": r["holdout_summary"].expectancy,
        "json_path": str(out_path),
    })

if index_rows:
    index_df = pd.DataFrame(index_rows).sort_values("holdout_profit_factor", ascending=False)
    index_df.to_csv(OUTPUT_DIR / "survivors_index.csv", index=False)
    display(index_df)
else:
    print("Nessuna regola ha superato la validazione hold-out con le soglie correnti "
          "(MIN_HOLDOUT_TRADES / MIN_HOLDOUT_PROFIT_FACTOR / MIN_HOLDOUT_WIN_RATE) — "
          "è un esito legittimo quanto trovarne: significa che nessun pattern scoperto in training "
          "ha retto sul periodo futuro censurato.")


scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-229.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-663.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-1705.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-1707.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-1709.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-2381.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-2466.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-2479.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-2484.json
scritto discovered_rules/ALPHA-ADAUSDC-1D-261001-2485.json


,rule_id,preset,direction,verdict_train,target_h,sell_pct,holdout_trades,holdout_profit_factor,holdout_win_rate,holdout_expectancy,json_path
1,ALPHA-ADAUSDC-1D-261001-663,balanced,short,PARTIAL-EDGE,48,0.1985,21,9999.0000,1.000000,0.124216,discovered_rules/ALPHA-ADAUSDC-1D-261001-663.json
3,ALPHA-ADAUSDC-1D-261001-1707,balanced,short,PARTIAL-EDGE,12,0.2293,5,9999.0000,1.000000,0.073216,discovered_rules/ALPHA-ADAUSDC-1D-261001-1707....
9,ALPHA-ADAUSDC-1D-261001-2485,balanced,short,PARTIAL-EDGE,12,0.2285,5,9999.0000,1.000000,0.075343,discovered_rules/ALPHA-ADAUSDC-1D-261001-2485....
4,ALPHA-ADAUSDC-1D-261001-1709,balanced,short,PARTIAL-EDGE,12,0.2317,5,9999.0000,1.000000,0.075983,discovered_rules/ALPHA-ADAUSDC-1D-261001-1709....
5,ALPHA-ADAUSDC-1D-261001-2381,balanced,short,PARTIAL-EDGE,12,0.2285,5,9999.0000,1.000000,0.075343,discovered_rules/ALPHA-ADAUSDC-1D-261001-2381....
7,ALPHA-ADAUSDC-1D-261001-2479,balanced,short,PARTIAL-EDGE,12,0.2284,5,9999.0000,1.000000,0.073036,discovered_rules/ALPHA-ADAUSDC-1D-261001-2479....
8,ALPHA-ADAUSDC-1D-261001-2484,balanced,short,PARTIAL-EDGE,12,0.2243,5,9999.0000,1.000000,0.074503,discovered_rules/ALPHA-ADAUSDC-1D-261001-2484....
2,ALPHA-ADAUSDC-1D-261001-1705,balanced,short,PARTIAL-EDGE,12,0.2285,6,2.7242,0.833333,0.038532,discovered_rules/ALPHA-ADAUSDC-1D-261001-1705....
0,ALPHA-ADAUSDC-1D-261001-229,balanced,short,PARTIAL-EDGE,12,0.2243,15,1.4983,0.600000,0.012106,discovered_rules/ALPHA-ADAUSDC-1D-261001-229.json
6,ALPHA-ADAUSDC-1D-261001-2466,balanced,short,PARTIAL-EDGE,12,0.2243,15,1.4983,0.600000,0.012106,discovered_rules/ALPHA-ADAUSDC-1D-261001-2466....


## 9. Riepilogo

- Regole tradeable scoperte in training (pre hold-out): vedi §6.
- Regole che hanno superato anche il hold-out (parametri congelati, nessuna riselezione): vedi §7-8.
- Ogni regola sopravvissuta è persistita in `discovered_rules/<rule_id>.json` con parametri operativi
  e performance complete; `discovered_rules/survivors_index.csv` è l'indice riassuntivo.

Per usare questo notebook in produzione su un vero database:

1. Impostare `DATA_SOURCE = "mysql"` e compilare `MYSQL_CONFIG` / `MYSQL_TABLE_BY_TIMEFRAME`.
2. Cambiare `TICKER` / `TIMEFRAME` (`"1H"` o `"1D"`) in §1.
3. Rivedere `PRESETS` e le soglie di hold-out in base alla lunghezza di storia realmente disponibile.
